In [ ]:
# Task01:  A warehouse robot needs to navigate from its starting position (S) to a target location (T) in a  grid-based environment. The robot faces the following constraints:
#  It can move only diagonally (top-left, top-right, bottom-left, bottom-right).
#  Movement cost follows Pythagoras’ theorem.
#  Some cells contain obstacles, which the robot must avoid.
#  The robot must find the shortest valid path to reach the target.
# 1. Define the CSP problem by specifying: o Variables o Domains o Constraints
# 2. Write a Python program using Google OR-Tools to solve the CSP problem and
# compute the shortest diagonal path.
# 3. Given a 5×5 grid where the robot starts at (1,1) and the target is at (4,4), solve the
# problem and output the path.

from ortools.sat.python import cp_model

# Create model
model = cp_model.CpModel()

# New grid (changed obstacles)
grid = [
    [0, 0, 1, 0, 0],
    [0, 0, 0, 1, 0],
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [0, 0, 0, 1, 0]
]

rows = len(grid)
cols = len(grid[0])

# Start and Target
start = (1, 1)
target = (4, 4)

K = 4  # number of steps (minimum for diagonal path)

# Variables 
row = [model.NewIntVar(0, rows - 1, f"row_{i}") for i in range(K)]
col = [model.NewIntVar(0, cols - 1, f"col_{i}") for i in range(K)]

# Constraints

# Start position
model.Add(row[0] == start[0])
model.Add(col[0] == start[1])

# Target position
model.Add(row[K - 1] == target[0])
model.Add(col[K - 1] == target[1])

# Diagonal movement constraints
for i in range(K - 1):

    move1 = model.NewBoolVar(f"up_left_{i}")     # (-1, -1)
    move2 = model.NewBoolVar(f"up_right_{i}")    # (-1, +1)
    move3 = model.NewBoolVar(f"down_left_{i}")   # (+1, -1)
    move4 = model.NewBoolVar(f"down_right_{i}")  # (+1, +1)

    # Only one move allowed
    model.Add(move1 + move2 + move3 + move4 == 1)

    # Apply moves
    model.Add(row[i+1] == row[i] - 1).OnlyEnforceIf(move1)
    model.Add(col[i+1] == col[i] - 1).OnlyEnforceIf(move1)

    model.Add(row[i+1] == row[i] - 1).OnlyEnforceIf(move2)
    model.Add(col[i+1] == col[i] + 1).OnlyEnforceIf(move2)

    model.Add(row[i+1] == row[i] + 1).OnlyEnforceIf(move3)
    model.Add(col[i+1] == col[i] - 1).OnlyEnforceIf(move3)

    model.Add(row[i+1] == row[i] + 1).OnlyEnforceIf(move4)
    model.Add(col[i+1] == col[i] + 1).OnlyEnforceIf(move4)


# Avoid Obstacles
for i in range(K):
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 1:
                model.AddForbiddenAssignments([row[i], col[i]], [(r, c)])

solver = cp_model.CpSolver()
status = solver.Solve(model)

# Output 
if status == cp_model.FEASIBLE or status == cp_model.OPTIMAL:
    print("Valid Path Found:\n")
    for i in range(K):
        print(f"Step {i}: ({solver.Value(row[i])}, {solver.Value(col[i])})")
else:
    print("No solution found")

Valid Path Found:

Step 0: (1, 1)
Step 1: (2, 2)
Step 2: (3, 3)
Step 3: (4, 4)


In [ ]:
# Task 02: A satellite monitors a remote island represented as a grid-based map, where:
#  1 indicates land   0 indicates water
# The goal is to track the largest continuous landmass and compute its perimeter to help predict erosion.
# 1. Define a constraint model where each cell is a binary variable (1 for land, 0 for water).
# 2. Identify boundary edges by checking adjacent land and water cells.
# 3. Solve the CSP model using Google OR-Tools to compute the perimeter efficiently.

from ortools.sat.python import cp_model

# Create model
model = cp_model.CpModel()

# Grid (1 = land, 0 = water)
grid = [
    [1, 1, 0, 0, 0],
    [1, 1, 0, 1, 1],
    [0, 0, 0, 1, 0],
    [1, 0, 0, 1, 1],
    [1, 1, 0, 0, 0]
]

rows = len(grid)
cols = len(grid[0])

# Variables

# Binary variables for each cell (1 = land, 0 = water)
cell = []
for i in range(rows):
    row_vars = []
    for j in range(cols):
        var = model.NewIntVar(0, 1, f"cell_{i}_{j}")
        row_vars.append(var)
    cell.append(row_vars)

# Constraints

# Fix variables according to given grid
for i in range(rows):
    for j in range(cols):
        model.Add(cell[i][j] == grid[i][j])

# Perimeter Calculation
perimeter_terms = []

for i in range(rows):
    for j in range(cols):

        if grid[i][j] == 1:  # only check land cells

            # Check 4 directions (up, down, left, right)

            # UP
            if i == 0 or grid[i-1][j] == 0:
                perimeter_terms.append(1)

            # DOWN
            if i == rows - 1 or grid[i+1][j] == 0:
                perimeter_terms.append(1)

            # LEFT
            if j == 0 or grid[i][j-1] == 0:
                perimeter_terms.append(1)

            # RIGHT
            if j == cols - 1 or grid[i][j+1] == 0:
                perimeter_terms.append(1)

# Total perimeter
perimeter = sum(perimeter_terms)

solver = cp_model.CpSolver()
status = solver.Solve(model)

# Output 
if status == cp_model.FEASIBLE or status == cp_model.OPTIMAL:
    print("Grid:")
    for row in grid:
        print(row)

    print("\nPerimeter of landmass:", perimeter)
else:
    print("No solution found")

Grid:
[1, 1, 0, 0, 0]
[1, 1, 0, 1, 1]
[0, 0, 0, 1, 0]
[1, 0, 0, 1, 1]
[1, 1, 0, 0, 0]

Perimeter of landmass: 28


In [ ]:
# Task 03: A salesperson must visit 10 cities, each exactly once, and return to the starting city. The goal is to minimize the total travel distance.
# 1. Define the CSP problem by specifying: o Variables o Domains
from ortools.sat.python import cp_model

# Create model
model = cp_model.CpModel()

# Number of cities
N = 10

# Distance matrix (custom, different from common examples)
dist  = [
        [0, 29, 20, 21, 16, 31, 100, 12, 4, 31],
        [29, 0, 15, 29, 28, 40, 72, 21, 29, 41],
        [20, 15, 0, 15, 14, 25, 81, 9, 23, 127],
        [21, 29, 15, 0, 4, 12, 92, 12, 25, 130],
        [16, 28, 14, 4, 0, 16, 94, 9, 102, 160],
        [31, 40, 25, 12, 16, 0, 95, 24, 36, 13],
        [10, 72, 81, 92, 94, 95, 0, 90, 10, 99],
        [120, 21, 9, 12, 99, 24, 90, 0, 15, 25],
        [4, 29, 23, 25, 20, 36, 101, 15, 0, 35],
        [31, 41, 27, 13, 160, 3, 99, 25, 35, 0],
    ]

# Variables

# X[i] = city at position i
X = [model.NewIntVar(0, N-1, f"city_{i}") for i in range(N)]

# Constraints

# All cities must be visited exactly once
model.AddAllDifferent(X)

# Cost Calculation

# Create cost variable
total_cost = model.NewIntVar(0, 1000, "total_cost")

cost_terms = []

for i in range(N - 1):
    # distance lookup
    d = model.NewIntVar(0, 100, f"d_{i}")
    model.AddElement(X[i] * N + X[i+1], sum(dist, []), d)
    cost_terms.append(d)

# Return to start
last_leg = model.NewIntVar(0, 100, "last_leg")
model.AddElement(X[N-1] * N + X[0], sum(dist, []), last_leg)
cost_terms.append(last_leg)

# Total cost
model.Add(total_cost == sum(cost_terms))

# Minimize distance
model.Minimize(total_cost)


solver = cp_model.CpSolver()
status = solver.Solve(model)

# Output

if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print("Optimal Path:\n")
    
    path = []
    for i in range(N):
        city = solver.Value(X[i])
        path.append(city)
        print(f"Step {i}: City {city}")
    
    print("\nReturn to start:", path[0])
    print("Total Distance:", solver.Value(total_cost))

else:
    print("No solution found")

Optimal Path:

Step 0: City 0
Step 1: City 9
Step 2: City 5
Step 3: City 3
Step 4: City 4
Step 5: City 7
Step 6: City 2
Step 7: City 1
Step 8: City 6
Step 9: City 8

Return to start: 0
Total Distance: 169
